# 01 案例一：2D 玩具数据扩散生成

这个案例用二维点代替图片。它训练很快，适合先把完整 DDPM 流程跑通：

1. 构造真实数据分布；
2. 随机选择时间步 `t`；
3. 前向加噪得到 `xt`；
4. 用 MLP 预测噪声；
5. 从纯噪声反向采样生成新点。

In [ ]:
import math
import os
import random
from pathlib import Path

import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
from tqdm.auto import tqdm

torch.manual_seed(42)
random.seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

In [ ]:
class DiffusionSchedule:
    def __init__(self, timesteps=300, beta_start=1e-4, beta_end=0.02, device=device):
        self.timesteps = timesteps
        self.betas = torch.linspace(beta_start, beta_end, timesteps, device=device)
        self.alphas = 1.0 - self.betas
        self.alpha_bars = torch.cumprod(self.alphas, dim=0)
        self.sqrt_alpha_bars = torch.sqrt(self.alpha_bars)
        self.sqrt_one_minus_alpha_bars = torch.sqrt(1.0 - self.alpha_bars)
        self.sqrt_recip_alphas = torch.sqrt(1.0 / self.alphas)
        self.posterior_variance = self.betas * (1.0 - torch.cat([torch.ones(1, device=device), self.alpha_bars[:-1]])) / (1.0 - self.alpha_bars)


def extract(values, t, x_shape):
    batch_size = t.shape[0]
    out = values.gather(0, t)
    return out.reshape(batch_size, *((1,) * (len(x_shape) - 1)))


def q_sample(x0, t, schedule, noise=None):
    if noise is None:
        noise = torch.randn_like(x0)
    return (
        extract(schedule.sqrt_alpha_bars, t, x0.shape) * x0
        + extract(schedule.sqrt_one_minus_alpha_bars, t, x0.shape) * noise
    )

In [ ]:
class SinusoidalPositionEmbeddings(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.dim = dim

    def forward(self, time):
        device = time.device
        half_dim = self.dim // 2
        embeddings = math.log(10000) / (half_dim - 1)
        embeddings = torch.exp(torch.arange(half_dim, device=device) * -embeddings)
        embeddings = time[:, None].float() * embeddings[None, :]
        return torch.cat((embeddings.sin(), embeddings.cos()), dim=-1)

In [ ]:
class MLPDenoiser(nn.Module):
    def __init__(self, data_dim=2, time_dim=64, hidden_dim=256, num_classes=None):
        super().__init__()
        self.time_mlp = nn.Sequential(
            SinusoidalPositionEmbeddings(time_dim),
            nn.Linear(time_dim, hidden_dim),
            nn.SiLU(),
        )
        self.label_emb = nn.Embedding(num_classes, hidden_dim) if num_classes is not None else None
        self.net = nn.Sequential(
            nn.Linear(data_dim + hidden_dim, hidden_dim),
            nn.SiLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.SiLU(),
            nn.Linear(hidden_dim, data_dim),
        )

    def forward(self, x, t, y=None):
        h = self.time_mlp(t)
        if self.label_emb is not None and y is not None:
            h = h + self.label_emb(y)
        return self.net(torch.cat([x, h], dim=1))

In [ ]:
@torch.no_grad()
def p_sample(model, x, t, t_index, schedule, y=None):
    betas_t = extract(schedule.betas, t, x.shape)
    sqrt_one_minus_alpha_bars_t = extract(schedule.sqrt_one_minus_alpha_bars, t, x.shape)
    sqrt_recip_alphas_t = extract(schedule.sqrt_recip_alphas, t, x.shape)

    predicted_noise = model(x, t, y) if y is not None else model(x, t)
    model_mean = sqrt_recip_alphas_t * (x - betas_t * predicted_noise / sqrt_one_minus_alpha_bars_t)

    if t_index == 0:
        return model_mean
    posterior_variance_t = extract(schedule.posterior_variance, t, x.shape)
    noise = torch.randn_like(x)
    return model_mean + torch.sqrt(posterior_variance_t) * noise


@torch.no_grad()
def sample(model, shape, schedule, y=None):
    model.eval()
    x = torch.randn(shape, device=device)
    for i in tqdm(reversed(range(schedule.timesteps)), total=schedule.timesteps):
        t = torch.full((shape[0],), i, device=device, dtype=torch.long)
        x = p_sample(model, x, t, i, schedule, y=y)
    return x

In [ ]:
def make_two_moons(n=20000, noise=0.06):
    angles = torch.rand(n) * math.pi
    moon1 = torch.stack([torch.cos(angles), torch.sin(angles)], dim=1)
    moon2 = torch.stack([1 - torch.cos(angles), 0.5 - torch.sin(angles)], dim=1)
    x = torch.cat([moon1[: n // 2], moon2[n // 2 :]], dim=0)
    x = x + noise * torch.randn_like(x)
    x = (x - x.mean(0)) / x.std(0)
    return x.float()

data = make_two_moons()
loader = DataLoader(TensorDataset(data), batch_size=512, shuffle=True)

plt.figure(figsize=(4, 4))
plt.scatter(data[:3000, 0], data[:3000, 1], s=3, alpha=0.5)
plt.axis("equal")
plt.title("Training data")
plt.show()

In [ ]:
schedule = DiffusionSchedule(timesteps=200)
model = MLPDenoiser(data_dim=2).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3)

epochs = 20
for epoch in range(epochs):
    model.train()
    total = 0.0
    for (x0,) in loader:
        x0 = x0.to(device)
        t = torch.randint(0, schedule.timesteps, (x0.shape[0],), device=device)
        noise = torch.randn_like(x0)
        xt = q_sample(x0, t, schedule, noise)
        pred_noise = model(xt, t)
        loss = F.mse_loss(pred_noise, noise)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total += loss.item()
    print(f"epoch {epoch + 1:02d} loss={total / len(loader):.4f}")

In [ ]:
generated = sample(model, (5000, 2), schedule).cpu()

plt.figure(figsize=(10, 4))
plt.subplot(1, 2, 1)
plt.scatter(data[:5000, 0], data[:5000, 1], s=2, alpha=0.5)
plt.axis("equal")
plt.title("Real")
plt.subplot(1, 2, 2)
plt.scatter(generated[:, 0], generated[:, 1], s=2, alpha=0.5)
plt.axis("equal")
plt.title("Generated")
plt.show()

可以尝试：
- 把 `timesteps` 从 200 改成 50 或 500；
- 把 MLP 的 `hidden_dim` 改小；
- 把数据改成圆环、高斯混合或螺旋线。